# Sampling, CLT & Hypothesis Testing

## What's covered

- **Law of Large Numbers** — sample averages converge to expectations
- **Central Limit Theorem** — the universal "averages are Gaussian" result
- **Monte Carlo integration** — using sampling to estimate expectations
- **Importance sampling** — variance reduction by sampling from a smarter distribution
- **Hypothesis testing** — the null/alternative framework, p-values, significance
- **A/B testing** — applied hypothesis testing for ML and product decisions
- **Type I / Type II errors** and **statistical power**
- Where this appears in ML — every mini-batch, every confidence interval, every A/B test


## Law of Large Numbers

The most fundamental theorem in probability: **sample averages converge to expectations.** Given i.i.d. random variables `X_1, X_2, \dots` with `\mathbb{E}[X] = \mu`:

$$
\bar{X}_n = \frac{1}{n} \sum_{i=1}^n X_i \quad \xrightarrow{n \to \infty} \quad \mu
$$

Two flavors of this result:

- **Weak LLN** — convergence in probability: `P(|\bar{X}_n - \mu| > \epsilon) \to 0` for any `\epsilon > 0`.
- **Strong LLN** — almost sure convergence: `\bar{X}_n \to \mu` with probability 1.

For ML, the distinction rarely matters. The takeaway: **estimate any expectation by averaging samples**.

**The LLN is what makes empirical risk minimization legitimate.** We minimize the training-set average loss `\frac{1}{n} \sum_i \ell(w; x_i, y_i)` as a proxy for the true expected loss `\mathbb{E}_{(x, y) \sim p}[\ell(w; x, y)]`. As `n` grows, the empirical average converges to the true expectation; under regularity conditions, the minimizer of the empirical loss converges to the minimizer of the true loss. *This is the only reason machine learning works at all.*

**The LLN says nothing about the rate.** How fast does `\bar{X}_n` converge to `\mu`? That's the next theorem.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

# Watch sample means converge to the true mean over n
true_mu = 5.0
sigma = 2.0

ns = np.unique(np.logspace(1, 5, 100).astype(int))
means = []
all_samples = rng.normal(true_mu, sigma, ns.max())
for n in ns:
    means.append(all_samples[:n].mean())

plt.figure(figsize=(8, 4))
plt.semilogx(ns, means, marker='.')
plt.axhline(true_mu, color='r', linestyle='--', label=f"true mean = {true_mu}")
plt.xlabel("n (sample size, log scale)"); plt.ylabel("sample mean")
plt.title("Law of Large Numbers — sample mean → true mean")
plt.legend(); plt.show()

print(f"Sample mean with n=10:      {all_samples[:10].mean():.4f}")
print(f"Sample mean with n=100:     {all_samples[:100].mean():.4f}")
print(f"Sample mean with n=10,000:  {all_samples[:10_000].mean():.4f}")


## The Central Limit Theorem — the universal Gaussian

The LLN tells you `\bar{X}_n \to \mu`. The **Central Limit Theorem** tells you *how fast*, and how the deviations from `\mu` are distributed.

**The statement.** For i.i.d. `X_1, X_2, \dots` with mean `\mu` and finite variance `\sigma^2`, the standardized sample mean converges in distribution to a standard normal:

$$
\frac{\sqrt{n} \, (\bar{X}_n - \mu)}{\sigma} \quad \xrightarrow{d} \quad \mathcal{N}(0, 1)
$$

Equivalently:

$$
\bar{X}_n \approx \mathcal{N}\!\left(\mu, \, \frac{\sigma^2}{n}\right) \quad \text{for large } n
$$

**Sample means are Gaussian, with standard error `\sigma / \sqrt{n}`.** This is the most-used theorem in applied statistics.

**Three remarkable features.**

- **Universality.** The original distribution of the `X_i` *does not matter*. As long as the variance is finite, the CLT holds. Sums of uniforms, exponentials, Bernoullis, you-name-it all become approximately Gaussian. This is *why* the Gaussian shows up so often in nature.
- **Rate.** Convergence is `O(1/\sqrt{n})` (in distributional distance). The Berry-Esseen theorem makes this precise.
- **Practical threshold.** As a rule of thumb, the CLT is "good enough" for `n \ge 30` if the underlying distribution isn't too pathological. For very skewed distributions (heavy tails), you may need much larger `n`.

**What the CLT does not say.** It tells you about *averages*, not individual `X_i`. Each `X_i` retains whatever ugly distribution it had. The CLT only smooths things out *when you average*. A single sample from a Cauchy distribution doesn't become Gaussian — but an *average* of 1000 finite-variance samples does.

**ML angle.** Stochastic gradient descent is the CLT in action. With batch size `B`, the mini-batch gradient is

$$
\hat{g}_B = \frac{1}{B} \sum_{i \in \text{batch}} \nabla \ell(w; x_i, y_i)
$$

an average of per-sample gradients. By the CLT, the noise in `\hat{g}_B` is approximately Gaussian with variance `\sigma^2 / B`. **Doubling the batch size halves the gradient-noise variance.** Standard error of every estimator in ML scales like `1/\sqrt{n}` — directly from the CLT.


In [ ]:
# Demonstrate CLT: averages of Exponential samples become Gaussian
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

ns = [1, 5, 30, 200]
for ax, n in zip(axes, ns):
    # Each "sample" is an average of n exponentials
    sample_means = rng.exponential(scale=1, size=(20_000, n)).mean(axis=1)
    ax.hist(sample_means, bins=60, density=True, alpha=0.7)

    # Overlay the Gaussian the CLT predicts
    mu, sigma = 1.0, 1.0
    xs = np.linspace(sample_means.min(), sample_means.max(), 200)
    ax.plot(xs, stats.norm.pdf(xs, mu, sigma / np.sqrt(n)),
            color='r', label="CLT Gaussian")
    ax.set_title(f"Average of n = {n} Exp(1)")
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()
print("The original Exponential is highly skewed, but the average becomes Gaussian as n grows.")


## Monte Carlo integration

LLN + CLT in one practical tool. To estimate `\mathbb{E}_{X \sim p}[g(X)]`, draw samples `X_1, \dots, X_n` from `p` and average `g`:

$$
\hat{I}_n = \frac{1}{n} \sum_{i=1}^n g(X_i)
$$

By the LLN, `\hat{I}_n \to \mathbb{E}[g(X)]`. By the CLT, the error is approximately Gaussian with standard deviation `\sigma_g / \sqrt{n}` where `\sigma_g^2 = \text{Var}(g(X))`. So the error rate is `O(1/\sqrt{n})`.

**Why MC is the default in high dimensions.** Grid-based numerical integration (Simpson's rule, etc.) has error rate `O(n^{-k/d})` for `k`-th order accuracy in `d` dimensions — which explodes as `d` grows. MC's `O(1/\sqrt{n})` is independent of dimension. In `d = 100`, MC is the only feasible option.

**Practical limitation.** The `O(1/\sqrt{n})` rate is *slow* — to reduce error by 10x, you need 100x more samples. For very precise estimates this becomes painful. Variance reduction techniques are how you fight back.


## Importance sampling — variance reduction

Sometimes drawing from `p` is hard, or wasteful (most samples contribute little). **Importance sampling** lets you draw from a *different* distribution `q` and reweight:

$$
\mathbb{E}_{X \sim p}[g(X)] = \mathbb{E}_{X \sim q}\!\left[g(X) \, \frac{p(X)}{q(X)}\right]
$$

The ratio `w(x) = p(x) / q(x)` is the **importance weight**. The estimator becomes:

$$
\hat{I}_n = \frac{1}{n} \sum_{i=1}^n g(X_i) w(X_i), \quad X_i \sim q
$$

**Why bother?**

- You can't sample from `p` directly, but you can sample from `q`.
- You want lower variance — choose `q` to put more mass where `|g(X) p(X)|` is large. The optimal `q^*(x) \propto |g(x)| p(x)` (Cauchy-Schwarz tells you this) would give zero variance — but it requires knowing the integral anyway. Practical `q`'s aim for "close to optimal."
- **Off-policy learning.** Your data comes from one policy, your target is a different one. Importance ratios convert between them.

**Watchouts.**

- If `q(x) = 0` where `p(x) > 0`, the estimator is biased (you'll miss mass that's in `p`).
- Heavy-tailed weights cause high variance — a few rare samples dominate the estimate. Self-normalized importance sampling (divide by sum of weights) often helps.
- In high dimensions, `p / q` ratios can blow up exponentially. Importance sampling stops working past moderate dimensions; you need MCMC instead.

**ML angle.** Off-policy reinforcement learning, PPO's clipped ratio objective, variational inference, RLHF — all involve importance ratios at the core.


In [ ]:
# Importance sampling: estimate E[X^2] for X ~ N(0, 1)
# (True value is 1)
#
# Bad q: sample from N(0, 0.1) — concentrated near 0, where x^2 is small. High variance.
# Good q: sample from N(0, 1) — equals p, so weights = 1, perfectly fine.
# Better q for E[X^2]: heavier-tailed than N(0, 1) — closer to the optimal q*.

N = 100_000

# (a) Naive MC: draw from p directly
samples_p = rng.normal(0, 1, N)
est_p = (samples_p ** 2).mean()
se_p  = (samples_p ** 2).std(ddof=1) / np.sqrt(N)

# (b) Importance sampling from q = N(0, 1.5) — heavier tails
sigma_q = 1.5
samples_q = rng.normal(0, sigma_q, N)
weights = stats.norm.pdf(samples_q, 0, 1) / stats.norm.pdf(samples_q, 0, sigma_q)
values = (samples_q ** 2) * weights
est_q = values.mean()
se_q  = values.std(ddof=1) / np.sqrt(N)

print(f"E[X^2] via naive MC from p:      {est_p:.4f}  ± {se_p:.4f}")
print(f"E[X^2] via importance from q=N(0, 1.5): {est_q:.4f}  ± {se_q:.4f}")
print(f"True value:                       1.0000")
print("\nBoth give correct answer; the right q can reduce variance.")


## Hypothesis testing — the null vs alternative framework

You want to decide whether model A is better than model B, or whether a feature has a real effect. Hypothesis testing is the formal frequentist framework.

**The setup.**

- **Null hypothesis `H_0`** — the "default" or "no effect" claim. Usually "the means are equal," "the coin is fair," "the new model is no better than the old one." We assume `H_0` is true unless the data strongly contradict it.
- **Alternative hypothesis `H_1`** — what you'd conclude if `H_0` is rejected. "The means differ," "the coin is biased," "the new model is better."
- **Test statistic** — a number computed from data that summarizes how surprising the data are under `H_0`. E.g., a t-statistic, a z-score, the difference in sample means.
- **p-value** — `P(\text{test statistic at least as extreme as observed} | H_0)`. The probability of seeing data this extreme *if `H_0` were true*. **Not** the probability that `H_0` is true.
- **Significance level `\alpha`** — pre-set threshold (typically 0.05). Reject `H_0` if p-value `< \alpha`.

**Two-sample test.** Most common in ML. Group A has `n_A` observations with sample mean `\bar{X}_A`; group B has `n_B` observations with sample mean `\bar{X}_B`. Test statistic (variance-pooled):

$$
T = \frac{\bar{X}_A - \bar{X}_B}{\sqrt{s_p^2 (1/n_A + 1/n_B)}}, \quad s_p^2 = \text{pooled sample variance}
$$

Under `H_0` and assuming approximate normality (CLT does this for us when `n` is large), `T` follows a `t`-distribution — or approximately `\mathcal{N}(0, 1)` for very large samples. Read off the p-value from the tails.

**Be careful with interpretation.**

- A small p-value means "if there were no effect, we'd rarely see data this extreme." It does *not* tell you the probability that the effect exists.
- A non-significant result (p > 0.05) doesn't mean "no effect" — only "we lack evidence to reject `H_0`." Absence of evidence ≠ evidence of absence.
- p-values are not effect sizes. A tiny but consistent difference can be statistically significant with enough data, yet practically meaningless.
- Multiple testing inflates false positives. If you run 20 tests at `\alpha = 0.05`, you expect 1 false positive even when nothing is real. Use Bonferroni, Holm, or false discovery rate (Benjamini-Hochberg) corrections.


In [ ]:
# A/B test simulation: did the new model improve click-through rate?

# Pretend control has CTR ~10%, treatment has CTR ~10.5%
n_control, n_treatment = 5000, 5000
p_control, p_treatment = 0.10, 0.105

control   = rng.random(n_control) < p_control
treatment = rng.random(n_treatment) < p_treatment

# Two-sample test of proportions (z-test approximation)
p1, p2 = control.mean(), treatment.mean()
diff = p2 - p1
# Pooled variance under H_0: p_pooled
p_pool = (control.sum() + treatment.sum()) / (n_control + n_treatment)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treatment))
z = diff / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))  # two-sided

print(f"Control CTR:    {p1:.4f}   (n = {n_control})")
print(f"Treatment CTR:  {p2:.4f}   (n = {n_treatment})")
print(f"Difference:     {diff:+.4f}")
print(f"z statistic:    {z:.4f}")
print(f"p-value:        {p_value:.4f}")
print(f"\nReject H_0 at α = 0.05? {'YES' if p_value < 0.05 else 'NO'}")
print("\nNote: with these sample sizes, we may not detect a 0.5pp effect.")
print("That's about statistical *power*, covered next.")


## Type I / Type II errors and statistical power

Every hypothesis test has four possible outcomes:

| | `H_0` true | `H_1` true |
|---|---|---|
| **Reject `H_0`** | **Type I error (false positive)** | Correct (true positive) |
| **Don't reject `H_0`** | Correct (true negative) | **Type II error (false negative)** |

- **Type I error rate** = `\alpha` (significance level) = probability of declaring an effect when there is none. Convention: 0.05.
- **Type II error rate** = `\beta` = probability of missing a real effect. Convention: 0.10 or 0.20.
- **Statistical power** = `1 - \beta` = probability of correctly detecting a real effect, given that it exists. We *want* high power (typically `\ge 0.8`).

**Power depends on four things:**

- **Sample size `n`** — bigger is better.
- **Effect size** — bigger differences are easier to detect.
- **Variance `\sigma^2`** — noisier data needs more samples.
- **Significance level `\alpha`** — being stricter (smaller `\alpha`) reduces power.

**Practical implication for A/B testing.** Before you run a test, decide what effect size you care about, then compute the required sample size to detect it with 80% power at `\alpha = 0.05`. Running an underpowered test wastes time and yields ambiguous results.

**Effect size cheat sheet for two-sample proportion tests.** Roughly, to detect an absolute difference of `\delta` in proportions around `p`:

$$
n_{\text{per arm}} \approx \frac{2 (z_{\alpha/2} + z_\beta)^2 p (1 - p)}{\delta^2}
$$

For `p = 0.10`, `\delta = 0.005` (5% relative lift), `\alpha = 0.05`, `\beta = 0.20`:
`n \approx 2 \cdot (1.96 + 0.84)^2 \cdot 0.10 \cdot 0.90 / 0.005^2 \approx 56{,}500` per arm. **Small effects require large samples.** This is the *Achilles heel* of A/B testing for marginal improvements.

**Bayesian alternative.** Bayesian A/B testing computes the posterior probability that B is better than A, often using Beta-Bernoulli conjugacy (from notebook 6). No p-values; direct probability statements. Increasingly common in modern data-science orgs.


In [ ]:
# Power analysis: how does sample size affect our ability to detect a small effect?
true_p_A, true_p_B = 0.10, 0.105
alpha = 0.05
N_simulations = 1000

print(f"Detecting a {true_p_B - true_p_A:.3f} absolute lift at α = {alpha}\n")
print(f"{'n per arm':>12} {'power':>10}")
for n in [1000, 5000, 10_000, 50_000, 100_000]:
    detections = 0
    for _ in range(N_simulations):
        A = rng.random(n) < true_p_A
        B = rng.random(n) < true_p_B
        p1, p2 = A.mean(), B.mean()
        p_pool = (A.sum() + B.sum()) / (2 * n)
        if p_pool == 0 or p_pool == 1:   # degenerate; pretend not detected
            continue
        se = np.sqrt(p_pool * (1 - p_pool) * (2/n))
        z = (p2 - p1) / se
        p_value = 2 * (1 - stats.norm.cdf(abs(z)))
        if p_value < alpha:
            detections += 1
    print(f"{n:>12,} {detections/N_simulations:>10.3f}")

print("\nFor very small effects, you need very large samples to achieve adequate power.")


## Where this appears in ML

This notebook closes the probability repo and bridges into the empirical, applied side of ML.

- **Mini-batch SGD.** The mini-batch gradient is a sample mean; its standard error scales like `1/\sqrt{B}` (CLT). Doubling batch size halves variance.
- **Monte Carlo integration.** ELBO estimation in VAEs, policy gradients (REINFORCE), Monte Carlo dropout for uncertainty, baseline subtraction for variance reduction.
- **Importance sampling.** Off-policy reinforcement learning (DQN replay buffers use forms of this), variational inference, RLHF policy reweighting.
- **MCMC.** Markov Chain Monte Carlo (Metropolis-Hastings, Gibbs, HMC, NUTS) — sampling from a posterior when conjugacy doesn't give you a closed form. Used in Bayesian deep learning, probabilistic programming.
- **Confidence intervals.** Bootstrap your model's accuracy 1000 times → distribution of accuracies → quantiles give you an interval estimate.
- **A/B testing of ML models.** Every recommendation, search, or ad-ranking system in production uses online A/B tests to decide if a new model is actually better.
- **Statistical significance of evaluations.** Don't claim model A beats model B without a paired test (e.g., paired bootstrap on per-sample metrics) and an effect size. Many "improvements" in papers are within noise.
- **Multi-armed bandits and Thompson sampling.** Bayesian or frequentist decisions about which arm to pull, formalized as sequential hypothesis testing.
- **Power analysis for experiments.** Before running an expensive online experiment, compute the required sample size for adequate power.
- **Bootstrap.** Resample with replacement to estimate the sampling distribution of an estimator without parametric assumptions.
- **Cross-validation as Monte Carlo.** K-fold CV is essentially MC over data partitions; the variance of the CV score is the variance of an average.
- **Calibration & ECE.** Reliability diagrams compare predicted probabilities vs empirical frequencies — implicitly comparing distributions via test-like procedures.
- **Conformal prediction.** Distribution-free prediction intervals with finite-sample frequentist coverage guarantees — built on order statistics.
- **Sequential testing in MLOps.** Group sequential designs, e-values, mSPRT — modern alternatives to fixed-`n` testing that let you peek at data without losing validity.

That closes the probability and statistics repo. From `P(\Omega) = 1` in notebook 1 to "we need 56,500 users per arm" in notebook 8.

**The three math foundation repos are complete:**

- **`linear-algebra`** — vectors, matrices, eigenvalues, SVD/PCA.
- **`calculus`** — derivatives, gradients, optimization, Taylor, integration.
- **`probability-and-statistics`** — distributions, inference, information theory, hypothesis testing.

Next on the roadmap: the **machine learning** repo finally builds on all three pillars.
